In [ ]:
%pip install python3-discogs-client

In [ ]:
import sys
import discogs_client
import pandas as pd
import os
import re
import tqdm
sys.path.append(os.path.abspath('../..'))

In [ ]:
from services.config import DISCOGS_TOKEN

In [ ]:
d = discogs_client.Client('D-Vella@github', user_token=DISCOGS_TOKEN)
d.set_timeout(5,5)
me=d.identity()

In [ ]:
# Source - https://stackoverflow.com/a/59085030
# Posted by Alberto, modified by community. See post 'Timeline' for change history
# Retrieved 2026-08-23, License - CC BY-SA 4.0

df = pd.DataFrame({'Artist': pd.Series(dtype='str'),
                   'Title': pd.Series(dtype='str'),
                   'PressingYear': pd.Series(dtype='int'),
                   'DateAdded': pd.Series(dtype='datetime64[ns]'),
                   'Format': pd.Series(dtype='str'),
                   'ReleaseID': pd.Series(dtype='int'),
                   'ReleaseMasterID': pd.Series(dtype='int')})


In [ ]:
total_items = len(me.collection_folders[0].releases)
print(f"Total items in collection: {total_items}")

In [ ]:
# Get all the items in my collection and put it into a dataframe
for item in tqdm.tqdm(me.collection_folders[0].releases):
    #seperate logic for items that need cleaning:
    ReleaseFormat = item.release.formats[0]
    ReleaseDesc = f"{ReleaseFormat['qty']} x {ReleaseFormat['name']} ({', '.join(ReleaseFormat['descriptions'])})"
    ReleaseMaster = item.release.master
    if ReleaseMaster:
        ReleaseYear = ReleaseMaster.year
        ReleaseMasterID = ReleaseMaster.id
    else:
        ReleaseYear = None
        ReleaseMasterID = f'M{item.release.id}'

    df.loc[len(df)] = {'Artist': re.sub(r'\s*\(\d+\)$', '', item.release.artists[0].name),
                       'Title': item.release.title,
                       'PressingYear': item.release.year if item.release.year != 0 else None,
                       'ReleaseYear': ReleaseYear,
                       'DateAdded': str(item.date_added)[0:10], #Just want the date.
                       'Format': item.release.formats[0]['name'],
                       'ReleaseID': item.release.id,
                       'ReleaseMasterID': ReleaseMasterID,
                       'ReleaseDesc': ReleaseDesc
}

In [ ]:
display(df)

In [ ]:
test = d.release(38018832)  
print(test)